# 01 Prepare Data

This notebook downloads and preprocesses the Goyal-Welch monthly data and Open Asset Pricing anomaly data, then saves the aligned target and predictor panels to `outputs/prepared`.


In [ ]:
# Optional for Colab / fresh environments:
# !pip install openassetpricing pyarrow statsmodels scikit-learn openpyxl seaborn


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from openassetpricing import OpenAP

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PREPARED_DIR = OUTPUT_DIR / "prepared"
for folder in [DATA_DIR, OUTPUT_DIR, PREPARED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

GW_XLSX_URL = "https://docs.google.com/spreadsheets/d/10_nkOkJPvq4eZgNl-1ys63PzhbnM3S2y/export?format=xlsx"
GW_SHEET_NAME = "Monthly"

ANOMALY_SOURCE = "vw"  # "vw" uses deciles_vw, "op" uses PredictorPortsFull
USE_PREBUILT_LS = True
SAMPLE_START = pd.Period("1970-01", freq="M")
RAW_START = SAMPLE_START - 1
FORECAST_START = pd.Period("1985-01", freq="M")

GW_EXCLUDED = {"cay", "pce", "ogap", "sntm", "fbm", "tchi", "shtint"}
GW_CANDIDATES = [
    "lty", "ltr", "tms", "tbl", "d/p", "e/p", "d/e", "b/m", "dfy", "dfr",
    "infl", "eqis", "ntis", "svar", "i/k", "csp", "vp", "impvar", "vrp",
    "govik", "lzrt", "skew", "crdstd", "wtexas", "accrul", "cfacc", "ndrbl",
    "skvw", "tail", "dtoy", "dtoat", "ygap", "rdsp", "rsvix", "gpce", "gip",
    "house", "avgcor", "disag"
]

LS_EXCLUDED = {"ConvDebt", "secureind", "IndIPO", "RDIPO"}


In [ ]:
def load_gw_monthly(xlsx_url: str, sheet_name: str) -> pd.DataFrame:
    return pd.read_excel(xlsx_url, sheet_name=sheet_name)


def prepare_gw_panel(raw_gw: pd.DataFrame) -> pd.DataFrame:
    df = raw_gw.copy()
    df["month"] = pd.to_datetime(df["yyyymm"].astype(str) + "01", format="%Y%m%d").dt.to_period("M")

    numeric_cols = ["ret", "Rfree"] + GW_CANDIDATES
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
    df["mkt_excess"] = df["ret"] - df["Rfree"]

    keep_cols = ["month", "ret", "Rfree", "mkt_excess"] + GW_CANDIDATES
    gw = (
        df.loc[df["month"] >= RAW_START, keep_cols]
        .drop_duplicates(subset=["month"])
        .sort_values("month")
        .set_index("month")
    )
    return gw


def load_anomaly_panel(source: str = "vw", use_prebuilt_ls: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:
    data_name = "deciles_vw" if source == "vw" else "op"
    raw = OpenAP().dl_port(data_name, "pandas")
    raw["month"] = pd.to_datetime(raw["date"]).dt.to_period("M")
    raw["port"] = raw["port"].astype(str)

    if use_prebuilt_ls and "LS" in set(raw["port"]):
        ls = raw.loc[raw["port"] == "LS", ["month", "signalname", "ret"]].copy()
    else:
        needed = raw.loc[raw["port"].isin(["01", "10"]), ["month", "signalname", "port", "ret"]].copy()
        wide = (
            needed.pivot_table(
                index=["month", "signalname"],
                columns="port",
                values="ret",
                aggfunc="first",
            )
            .rename_axis(columns=None)
            .reset_index()
        )
        wide["ret"] = wide["10"] - wide["01"]
        ls = wide.loc[:, ["month", "signalname", "ret"]]

    wide_ls = (
        ls.pivot_table(index="month", columns="signalname", values="ret", aggfunc="first")
        .sort_index()
    )
    wide_ls = wide_ls.loc[wide_ls.index >= RAW_START]

    first_month = wide_ls.apply(lambda col: col.dropna().index.min())
    metadata = pd.DataFrame(
        {
            "signalname": wide_ls.columns,
            "first_month": first_month.reindex(wide_ls.columns).values,
            "missing_count_raw": wide_ls.isna().sum().values,
        }
    ).set_index("signalname")

    keep = [
        signal
        for signal in wide_ls.columns
        if signal not in LS_EXCLUDED and metadata.loc[signal, "first_month"] <= FORECAST_START
    ]
    filtered = wide_ls.loc[:, keep].copy()
    missing_before = int(filtered.isna().sum().sum())
    filtered = filtered.apply(lambda row: row.fillna(row.mean()), axis=1)
    filtered = filtered / 100.0
    metadata = metadata.loc[keep].copy()
    metadata["missing_count_after_filter"] = filtered.isna().sum().reindex(metadata.index).values

    print(
        f"Anomaly source '{source}' keeps {len(keep)} signals; "
        f"missing cells before monthly mean imputation: {missing_before:,}."
    )
    return filtered, metadata


def select_complete_gw_predictors(gw_panel: pd.DataFrame, common_end: pd.Period) -> list[str]:
    sample = gw_panel.loc[SAMPLE_START:common_end]
    keep = []
    for col in GW_CANDIDATES:
        if col in GW_EXCLUDED:
            continue
        if sample[col].notna().all():
            keep.append(col)
    return keep


def build_model_frame(target_panel: pd.DataFrame, features: pd.DataFrame) -> pd.DataFrame:
    lagged_features = features.shift(1)
    model = pd.concat([target_panel.loc[:, ["mkt_excess", "Rfree"]], lagged_features], axis=1)
    model = model.dropna()
    return model.loc[model.index >= SAMPLE_START]


In [ ]:
raw_gw = load_gw_monthly(GW_XLSX_URL, GW_SHEET_NAME)
gw_panel = prepare_gw_panel(raw_gw)
ls_panel, ls_metadata = load_anomaly_panel(ANOMALY_SOURCE, USE_PREBUILT_LS)

common_index = gw_panel.index.intersection(ls_panel.index)
common_end = common_index.max()

gw_predictors = select_complete_gw_predictors(gw_panel, common_end)
target_panel = gw_panel.loc[common_index, ["mkt_excess", "Rfree"]].rename(columns={"Rfree": "Rfree"})
gw_features = gw_panel.loc[common_index, gw_predictors]
ls_features = ls_panel.loc[common_index]

gw_model = build_model_frame(target_panel, gw_features)
ls_model = build_model_frame(target_panel, ls_features)

shared_sample_index = gw_model.index.intersection(ls_model.index)
gw_model = gw_model.loc[shared_sample_index]
ls_model = ls_model.loc[shared_sample_index]

prepared_objects = {
    "target_panel": target_panel.loc[shared_sample_index],
    "gw_features": gw_features.loc[shared_sample_index],
    "ls_features": ls_features.loc[shared_sample_index],
    "gw_model": gw_model,
    "ls_model": ls_model,
    "ls_metadata": ls_metadata,
}

for name, df in prepared_objects.items():
    csv_path = PREPARED_DIR / f"{name}.csv"
    parquet_path = PREPARED_DIR / f"{name}.parquet"
    df.to_csv(csv_path)
    df.to_parquet(parquet_path)

summary = pd.DataFrame(
    {
        "object": ["target_panel", "gw_features", "ls_features", "gw_model", "ls_model"],
        "rows": [len(prepared_objects[k]) for k in ["target_panel", "gw_features", "ls_features", "gw_model", "ls_model"]],
        "cols": [prepared_objects[k].shape[1] for k in ["target_panel", "gw_features", "ls_features", "gw_model", "ls_model"]],
    }
)
print(f"Prepared data saved to: {PREPARED_DIR}")
print(f"Common target-month sample: {shared_sample_index.min()} to {shared_sample_index.max()} ({len(shared_sample_index)} months)")
print(f"GW predictors kept after exclusions + completeness filter: {len(gw_predictors)}")
print(f"LS anomaly predictors kept after exclusions + start-by-1985 filter: {ls_features.shape[1]}")
summary


Saved artifacts:

- `target_panel`
- `gw_features`
- `ls_features`
- `gw_model`
- `ls_model`
- `ls_metadata`

Each is written as both CSV and parquet under `outputs/prepared`.
